# Ranga — Synthetic Dataset Generator

**MLX-local generation on Apple Silicon using Gemma 4 12B**

This notebook builds three **function-calling** training files for Ranga. The model never memorises hospital names or copay amounts — it learns to call tools that query the live database (simulated here from GeoJSON + markdown rules).

| Output file | Records | Purpose |
|---|---|---|
| `ranga_sft_500.jsonl` | 500 | Multi-turn tool-use trajectories for SFT |
| `ranga_dpo_200.jsonl` | 200 | Chosen vs rejected tool-use trajectories for DPO |
| `ranga_eval_50.jsonl` | 50 | Eval queries with expected tool-call sequences |
| `ranga_tools.json` | 1 | Tool schemas (shared across all records) |

## Why function calling?

Hospital networks and insurance rules change via background JSON delta sync. The on-device model must:
1. **Call tools** to look up providers, coverage, and costs at runtime
2. **Never hallucinate** provider names or copay figures from training memory
3. **Chain calls** — search → verify coverage → calculate OOP → respond

## Model: Gemma 4 12B Unified

- **11.95B parameters**, encoder-free multimodal, **256K context**
- Native function calling via `<|tool_call>` / `<|tool_response>` tokens
- MLX weights: `mlx-community/gemma-4-12B-it-4bit` (~6.7 GB)

## Source data

| Source | File | Role |
|---|---|---|
| Hospitals (dynamic DB) | `health_facilities.geojson` | Provider records synced to SQLite |
| Insurance rules | `InPatient-Medical-Insurance-Contract.md` | Britam policy prose |
| Insurance rules | `Old_Mutual_Insurance_Rwanda_Medical_Insurance_Tariff.md` | Eden Care / Old Mutual tariff |
| Complaint seeds | [MT Samples](https://huggingface.co/datasets/harishnair04/mtsamples) | Natural-language query seeds |

## 0 — Install dependencies

Run once, then restart the kernel.

In [ ]:
!pip install mlx-vlm datasets pandas tqdm --quiet

## 1 — Config

In [ ]:
import json, math, random, re, pathlib, textwrap, copy
from datetime import datetime

DATASET_DIR   = pathlib.Path(".")
GEOJSON_PATH  = DATASET_DIR / "health_facilities.geojson"
BRITAM_MD     = DATASET_DIR / "InPatient-Medical-Insurance-Contract.md"
OLD_MUTUAL_MD = DATASET_DIR / "Old_Mutual_Insurance_Rwanda_Medical_Insurance_Tariff.md"

OUT_DIR    = DATASET_DIR / "ranga_output"
OUT_DIR.mkdir(exist_ok=True)
SFT_PATH   = OUT_DIR / "ranga_sft_500.jsonl"
DPO_PATH   = OUT_DIR / "ranga_dpo_200.jsonl"
EVAL_PATH  = OUT_DIR / "ranga_eval_50.jsonl"
TOOLS_PATH = OUT_DIR / "ranga_tools.json"

MODEL_ID    = "mlx-community/gemma-4-12B-it-4bit"
MAX_TOKENS  = 512
TEMPERATURE = 0.7
SEED        = 42

ALU_LAT, ALU_LON = -1.9695, 30.1589
MAX_RADIUS_KM    = 25
TOP_PROVIDERS    = 15

N_TEMPLATE = 300
N_CUSTOM   = 200
N_DPO      = 200
N_EVAL     = 50

random.seed(SEED)
print(f"Output: {OUT_DIR.resolve()}")

## 2 — Load Gemma 4 12B

Used only for generating natural-language **user queries** and **final responses**. Tool calls and tool results are deterministic.

In [ ]:
from mlx_vlm import load, generate

print(f"Loading {MODEL_ID} …")
model, processor = load(MODEL_ID)
print("Model ready.")

def llm(prompt: str, max_tokens: int = MAX_TOKENS, temp: float = TEMPERATURE) -> str:
    messages = [{"role": "user", "content": prompt}]
    formatted = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    return generate(model, processor, prompt=formatted, max_tokens=max_tokens, temp=temp, verbose=False)

## 3 — Load and filter MT Samples

In [ ]:
from datasets import load_dataset
import pandas as pd

raw = load_dataset("harishnair04/mtsamples", split="train")
df  = raw.to_pandas()
print(f"Loaded {len(df)} rows")
df.head(2)

In [ ]:
SPECIALTY_MAP = {
    "Allergy / Immunology": "General Consultation", "Cardiovascular / Pulmonary": "General Consultation",
    "Dentistry": "Dental Care", "Dermatology": "General Consultation",
    "Emergency Room Reports": "Emergency Services", "ENT - Otolaryngology": "General Consultation",
    "Gastroenterology": "Laboratory Tests", "General Medicine": "General Consultation",
    "Hematology - Oncology": "Laboratory Tests", "Neurology": "General Consultation",
    "Obstetrics / Gynecology": "General Consultation", "Ophthalmology": "Optical Care",
    "Orthopedic": "Physical Therapy", "Pain Management": "Physical Therapy",
    "Pathology": "Laboratory Tests", "Pediatrics - Neonatal": "General Consultation",
    "Physical Medicine - Rehab": "Physical Therapy", "Podiatry": "General Consultation",
    "Psychiatry / Psychology": "General Consultation", "Radiology": "Laboratory Tests",
    "Rheumatology": "General Consultation", "Sleep Medicine": "General Consultation",
    "Speech - Language": "Physical Therapy", "Urology": "General Consultation",
}
DISCARD = {"Autopsy", "Neurosurgery", "Surgery", "Bariatrics"}
CLINICAL_BLOCKLIST = re.compile(
    r'\b(diagnosis|diagnos\w+|prescri\w+|medic\w+|drug|tablet|capsule|mg|ml|dose|'
    r'injection|iv |surgery|surgical|procedure|biopsy|catheter|resect\w+|'
    r'transplant|bypass|suture|incision|abscess|carcinoma|malignant|'
    r'hypertension|diabetes|atrial|fibrillation|edema|sepsis)\b', re.IGNORECASE)
CRISIS_KEYWORDS = re.compile(r'\b(suicid\w+|self.harm|kill myself|want to die|hopeless)\b', re.IGNORECASE)

def extract_context(row) -> dict | None:
    specialty = str(row.get("medical_specialty", "")).strip()
    if specialty in DISCARD or specialty not in SPECIALTY_MAP:
        return None
    desc = str(row.get("description", "")).strip()
    if len(desc) < 20 or CRISIS_KEYWORDS.search(desc):
        return None
    clean_desc = CLINICAL_BLOCKLIST.sub("", desc).strip()
    clean_desc = re.sub(r'\s{2,}', ' ', clean_desc)
    transcription = str(row.get("transcription", ""))[:200]
    age_match = re.search(r'(\d+)[- ]year[- ]old', transcription, re.IGNORECASE)
    age_band = "adult"
    if age_match:
        age = int(age_match.group(1))
        age_band = "child" if age < 18 else ("elderly" if age >= 65 else "adult")
    return {
        "service_category": SPECIALTY_MAP[specialty],
        "age_band": age_band,
        "complaint_seed": clean_desc[:200],
        "has_chronic": bool(re.search(r'history of|chronic|recurrent', desc, re.IGNORECASE)),
        "source_row_id": int(row.name),
    }

contexts = [c for _, row in df.iterrows() if (c := extract_context(row)) is not None]
print(f"Usable contexts: {len(contexts)}")

## 4 — Source A: Hospitals from GeoJSON

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def feature_centroid(geometry: dict) -> tuple[float, float] | tuple[None, None]:
    gtype = geometry["type"]
    if gtype == "Point":
        return geometry["coordinates"]
    if gtype == "Polygon":
        ring = geometry["coordinates"][0]
        return sum(c[0] for c in ring)/len(ring), sum(c[1] for c in ring)/len(ring)
    return None, None

AMENITY_TO_TIER = {
    "pharmacy": "Tier 1 — Pharmacy", "clinic": "Tier 1 — Health Centre",
    "doctors": "Tier 1 — Health Centre", "hospital": "Tier 2 — District Hospital",
}
TIER3_KEYWORDS = re.compile(r'king faisal|military|district hospital|teaching|oncology|cancer', re.I)
SERVICE_AMENITY = {
    "pharmacy": ["Pharmacy & Medication"],
    "clinic": ["General Consultation", "Laboratory Tests"],
    "doctors": ["General Consultation"],
    "hospital": ["General Consultation", "Dental Care", "Optical Care",
                 "Laboratory Tests", "Emergency Services", "Physical Therapy"],
}

with open(GEOJSON_PATH) as f:
    geo = json.load(f)

rows = []
for feat in geo["features"]:
    props = feat["properties"]
    name = (props.get("name") or props.get("name_en") or "").strip()
    if not name:
        continue
    amenity = props.get("amenity") or props.get("healthcare") or ""
    if amenity not in SERVICE_AMENITY or props.get("adm1_name") != "Kigali City":
        continue
    lon, lat = feature_centroid(feat["geometry"])
    if lon is None:
        continue
    dist = haversine_km(ALU_LAT, ALU_LON, lat, lon)
    if dist > MAX_RADIUS_KM:
        continue
    tier = AMENITY_TO_TIER.get(amenity, "Tier 2 — District Hospital")
    if TIER3_KEYWORDS.search(name):
        tier = "Tier 3 — Referral Hospital"
    rows.append({
        "provider_id": props["id"], "name": name,
        "physical_address": f"{props.get('adm3_name','')}, {props.get('adm2_name','')}, Kigali",
        "distance_from_alu_km": round(dist, 2), "phone_number": "",
        "amenity": amenity, "tier_level": tier, "latitude": lat, "longitude": lon, "active": 1,
    })

hospitals = (
    pd.DataFrame(rows).sort_values("distance_from_alu_km")
    .drop_duplicates("name").head(TOP_PROVIDERS).reset_index(drop=True)
)
print(f"Providers within {MAX_RADIUS_KM} km: {len(hospitals)}")
hospitals[["name", "tier_level", "distance_from_alu_km"]].head(8)

## 5 — Source B: Insurance rules from Markdown

In [ ]:
def load_markdown_sources() -> dict[str, str]:
    return {"britam": BRITAM_MD.read_text(), "old_mutual": OLD_MUTUAL_MD.read_text()}

def parse_britam_rules(text: str) -> dict:
    return {
        "scheme_name": "Britam", "source_file": BRITAM_MD.name,
        "inpatient_no_copay": bool(re.search(r'100%\s*NO\s*CO-?PAY', text, re.I)),
        "default_copayment_rate": 0.10, "requires_referral_tier3": True,
        "requires_pre_auth_elective": "pre-authoris" in text.lower(),
        "associated_provider_only": "Associated Provider" in text,
    }

def parse_old_mutual_rules(text: str) -> dict:
    heza = re.search(r'HEZA\s*CARE.*?OUTPATIENT.*?(\d[\d,]+)\s*-\s*(\d[\d,]+)', text, re.I|re.S)
    return {
        "scheme_name": "EdenCare", "source_file": OLD_MUTUAL_MD.name,
        "default_copayment_rate": 0.15, "deductible_rwf": 5_000,
        "requires_pre_auth_specialist": True,
        "heza_outpatient_limit_rwf": tuple(int(x.replace(",","")) for x in heza.groups()) if heza else (250_000, 450_000),
    }

PUBLIC_SCHEMES = {
    "CBHI": {"scheme_name": "CBHI", "source_file": "MOH referral guidelines",
             "default_copayment_rate": 0.10, "requires_referral": True},
    "RSSB": {"scheme_name": "RSSB", "source_file": "RSSB scheme rules",
             "default_copayment_rate": 0.15, "requires_referral_tier3": True},
}
BASE_FEES_RWF = {
    "General Consultation": 15_000, "Dental Care": 25_000, "Optical Care": 30_000,
    "Laboratory Tests": 20_000, "Pharmacy & Medication": 10_000,
    "Emergency Services": 50_000, "Physical Therapy": 20_000,
}
OUT_OF_NETWORK_NAMES = {"Legacy Hospital", "Legacy Clinics", "UCL Clinic"}
SCHEMES = ["CBHI", "RSSB", "EdenCare", "Britam"]
SERVICES = list(BASE_FEES_RWF.keys())

docs = load_markdown_sources()
britam_meta, eden_meta = parse_britam_rules(docs["britam"]), parse_old_mutual_rules(docs["old_mutual"])

def build_insurance_rules(hospitals_df: pd.DataFrame) -> pd.DataFrame:
    records, rule_id = [], 1
    for _, prov in hospitals_df.iterrows():
        is_legacy = prov["name"] in OUT_OF_NETWORK_NAMES
        tier, amenity_services = prov["tier_level"], SERVICE_AMENITY.get(prov["amenity"], SERVICES)
        for scheme, meta in [("CBHI", PUBLIC_SCHEMES["CBHI"]), ("RSSB", PUBLIC_SCHEMES["RSSB"]),
                             ("EdenCare", eden_meta), ("Britam", britam_meta)]:
            for service in SERVICES:
                if service not in amenity_services and prov["amenity"] != "hospital":
                    continue
                copay = meta["default_copayment_rate"]
                if scheme == "Britam" and meta.get("inpatient_no_copay") and service == "Emergency Services":
                    copay = 0.0
                records.append({
                    "rule_id": rule_id, "provider_id": prov["provider_id"], "scheme_name": scheme,
                    "service_category": service, "tier_level": tier, "copayment_rate": copay,
                    "requires_referral": int("Tier 3" in tier or meta.get("requires_referral") or meta.get("requires_referral_tier3", False)),
                    "requires_pre_auth": int((scheme in ("EdenCare","Britam") and "Tier 3" in tier) or meta.get("requires_pre_auth_specialist", False)),
                    "out_of_network": int(is_legacy and scheme in ("EdenCare", "Britam")),
                    "base_fee_rwf": BASE_FEES_RWF[service], "deductible_rwf": meta.get("deductible_rwf", 0),
                    "source_file": meta.get("source_file", ""), "active": 1,
                })
                rule_id += 1
    return pd.DataFrame(records)

rules = build_insurance_rules(hospitals)
rules_full = rules.merge(hospitals[["provider_id","name","physical_address","distance_from_alu_km"]], on="provider_id")
rules_in_network = rules_full[(rules_full["active"]==1) & (rules_full["out_of_network"]==0)].reset_index(drop=True)
print(f"Rules: {len(rules_full)} total, {len(rules_in_network)} in-network")

## 6 — Tool schemas + runtime executor

These tools mirror the Ranga app's SQLite query layer. During training we **simulate** them with the DataFrames above; at inference the same function names hit the on-device database (updated via JSON delta sync).

In [ ]:
RANGA_TOOLS = [
    {"type": "function", "function": {
        "name": "search_covered_providers",
        "description": "Search in-network providers for a scheme and service near ALU campus. Returns live DB records — never guess provider names.",
        "parameters": {"type": "object", "properties": {
            "scheme": {"type": "string", "enum": SCHEMES},
            "service": {"type": "string", "enum": SERVICES},
            "max_distance_km": {"type": "number", "description": "Radius from ALU campus", "default": 25},
            "limit": {"type": "integer", "description": "Max results", "default": 3},
        }, "required": ["scheme", "service"]},
    }},
    {"type": "function", "function": {
        "name": "get_provider_coverage",
        "description": "Check whether a specific provider is in-network and what tier/referral rules apply.",
        "parameters": {"type": "object", "properties": {
            "provider_id": {"type": "string"},
            "scheme": {"type": "string", "enum": SCHEMES},
            "service": {"type": "string", "enum": SERVICES},
        }, "required": ["provider_id", "scheme", "service"]},
    }},
    {"type": "function", "function": {
        "name": "calculate_oop_cost",
        "description": "Calculate out-of-pocket cost in RWF. Must be called after confirming provider coverage.",
        "parameters": {"type": "object", "properties": {
            "provider_id": {"type": "string"},
            "scheme": {"type": "string", "enum": SCHEMES},
            "service": {"type": "string", "enum": SERVICES},
            "referral_followed": {"type": "boolean"},
            "chronic_bypass": {"type": "boolean", "default": False},
        }, "required": ["provider_id", "scheme", "service", "referral_followed"]},
    }},
]

with open(TOOLS_PATH, "w") as f:
    json.dump(RANGA_TOOLS, f, indent=2)
print(f"Saved {len(RANGA_TOOLS)} tool schemas to {TOOLS_PATH.name}")
for t in RANGA_TOOLS:
    print(f"  • {t['function']['name']}")

In [ ]:
def compute_oop(rule: dict, referral_followed: bool = True) -> int:
    fee, copay, scheme, oon = float(rule["base_fee_rwf"]), float(rule["copayment_rate"]), str(rule["scheme_name"]), int(rule["out_of_network"])
    if oon == 1: return int(fee)
    if scheme == "CBHI": return int(fee) if not referral_followed else int(fee * copay)
    if scheme == "RSSB": return int(fee * copay)
    return int(max(float(rule.get("deductible_rwf", 0)), fee * copay))

def _get_rule(provider_id: str, scheme: str, service: str) -> dict | None:
    mask = (rules_full["provider_id"]==provider_id) & (rules_full["scheme_name"]==scheme) & (rules_full["service_category"]==service) & (rules_full["active"]==1)
    row = rules_full[mask]
    return None if row.empty else row.iloc[0].to_dict()

def tool_search_covered_providers(scheme: str, service: str, max_distance_km: float = 25, limit: int = 3) -> dict:
    mask = ((rules_in_network["scheme_name"]==scheme) & (rules_in_network["service_category"]==service) &
            (rules_in_network["distance_from_alu_km"] <= max_distance_km))
    hits = rules_in_network[mask].sort_values("distance_from_alu_km").head(limit)
    providers = [{"provider_id": r["provider_id"], "name": r["name"], "tier_level": r["tier_level"],
                  "distance_km": r["distance_from_alu_km"], "address": r["physical_address"],
                  "requires_referral": bool(r["requires_referral"]), "requires_pre_auth": bool(r["requires_pre_auth"])}
                 for _, r in hits.iterrows()]
    return {"scheme": scheme, "service": service, "count": len(providers), "providers": providers}

def tool_get_provider_coverage(provider_id: str, scheme: str, service: str) -> dict:
    rule = _get_rule(provider_id, scheme, service)
    if rule is None:
        return {"provider_id": provider_id, "found": False, "in_network": False}
    return {"provider_id": provider_id, "name": rule["name"], "found": True,
            "in_network": rule["out_of_network"]==0, "tier_level": rule["tier_level"],
            "requires_referral": bool(rule["requires_referral"]), "requires_pre_auth": bool(rule["requires_pre_auth"]),
            "copayment_rate": rule["copayment_rate"], "base_fee_rwf": rule["base_fee_rwf"]}

def tool_calculate_oop_cost(provider_id: str, scheme: str, service: str,
                            referral_followed: bool, chronic_bypass: bool = False) -> dict:
    rule = _get_rule(provider_id, scheme, service)
    if rule is None:
        return {"error": "provider_not_found"}
    effective_referral = referral_followed or chronic_bypass
    oop = compute_oop(rule, referral_followed=effective_referral)
    return {"provider_id": provider_id, "scheme": scheme, "service": service,
            "oop_rwf": oop, "copayment_rate": rule["copayment_rate"],
            "base_fee_rwf": rule["base_fee_rwf"], "referral_followed": effective_referral,
            "chronic_bypass_applied": chronic_bypass}

TOOL_DISPATCH = {
    "search_covered_providers": tool_search_covered_providers,
    "get_provider_coverage": tool_get_provider_coverage,
    "calculate_oop_cost": tool_calculate_oop_cost,
}

def execute_tool(name: str, arguments: dict) -> dict:
    return TOOL_DISPATCH[name](**arguments)

# Sanity check
demo = tool_search_covered_providers("CBHI", "Dental Care", limit=2)
print(json.dumps(demo, indent=2)[:400], "…")

## 7 — Trajectory builder

Each training record is a **multi-turn conversation** with tool calls. The model learns the pattern:

```
user query → tool_call(search) → tool_result → tool_call(calculate_oop) → tool_result → natural answer
```

When the user mentions a rejected prior visit, we add `get_provider_coverage` first.

In [ ]:
def _tool_call(name: str, arguments: dict) -> dict:
    return {"type": "function", "function": {"name": name, "arguments": arguments}}

def _tool_msg(call_id: str, name: str, result: dict) -> dict:
    return {"role": "tool", "tool_call_id": call_id, "name": name, "content": json.dumps(result, ensure_ascii=False)}

def _assistant_tool_turn(call_id: str, name: str, arguments: dict, result: dict) -> list[dict]:
    return [
        {"role": "assistant", "content": None, "tool_calls": [{"id": call_id, **_tool_call(name, arguments)}]},
        _tool_msg(call_id, name, result),
    ]

SYSTEM_PROMPT = textwrap.dedent("""\
    You are Ranga, an offline hospital recommendation assistant for Rwanda.
    You MUST use the provided tools to look up providers, check coverage, and
    calculate costs. Never guess provider names, network status, or copay amounts.
    Hospital and insurance data changes dynamically — always query the database.

    Safety rules:
    1. NEVER diagnose a medical condition.
    2. NEVER name a drug, medication, or dosage.
    3. NEVER recommend out-of-network providers.
    4. ALWAYS call calculate_oop_cost before stating a price.
    5. Mental health crisis → respond only: EMERGENCY_SOS
    """).strip()

def build_final_response(rule: dict, oop: int, has_referral: bool, bypass: bool) -> str:
    scheme, service, name = rule["scheme_name"], rule["service_category"], rule["name"]
    dist, tier = rule["distance_from_alu_km"], rule["tier_level"]
    copay_pct = int(float(rule["copayment_rate"]) * 100)
    pre_auth = int(rule["requires_pre_auth"]) == 1
    referral = int(rule["requires_referral"]) == 1
    lines = []
    if bypass:
        lines.append(f"Your chronic visit history lets you bypass the referral requirement for {scheme}.")
    elif referral and not has_referral:
        lines.append(f"Get a referral from your local health post before visiting a {tier.lower()} facility.")
    lines.append(f"{name} ({dist:.1f} km from ALU) covers {service} under {scheme}.")
    lines.append(f"Your copayment is {copay_pct}% — {oop:,} RWF out of pocket.")
    if pre_auth:
        lines.append(f"Obtain pre-authorisation from {scheme} before your visit.")
    lines.append("Bring your insurance card.")
    return " ".join(lines)

def build_tool_trajectory(
    user_query: str,
    rule: dict,
    has_referral: bool,
    has_chronic: bool,
    prior_failure_provider_id: str | None = None,
) -> tuple[list[dict], list[dict]]:
    """Return (messages, tool_call_log) for a complete tool-use trajectory."""
    scheme, service = rule["scheme_name"], rule["service_category"]
    bypass = has_chronic and int(rule["requires_referral"]) == 1
    referral_followed = has_referral or bypass
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_query}]
    tool_log, step = [], 0

    if prior_failure_provider_id:
        step += 1
        cid = f"call_{step}"
        args = {"provider_id": prior_failure_provider_id, "scheme": scheme, "service": service}
        result = execute_tool("get_provider_coverage", args)
        tool_log.append({"call": args, "result": result})
        messages.extend(_assistant_tool_turn(cid, "get_provider_coverage", args, result))

    step += 1
    cid = f"call_{step}"
    search_args = {"scheme": scheme, "service": service, "max_distance_km": MAX_RADIUS_KM, "limit": 3}
    search_result = execute_tool("search_covered_providers", search_args)
    tool_log.append({"call": search_args, "result": search_result})
    messages.extend(_assistant_tool_turn(cid, "search_covered_providers", search_args, search_result))

    step += 1
    cid = f"call_{step}"
    oop_args = {"provider_id": rule["provider_id"], "scheme": scheme, "service": service,
                "referral_followed": referral_followed, "chronic_bypass": bypass}
    oop_result = execute_tool("calculate_oop_cost", oop_args)
    tool_log.append({"call": oop_args, "result": oop_result})
    messages.extend(_assistant_tool_turn(cid, "calculate_oop_cost", oop_args, oop_result))

    final = build_final_response(rule, oop_result["oop_rwf"], has_referral, bypass)
    messages.append({"role": "assistant", "content": final})
    return messages, tool_log

def pick_rule(service_category: str, scheme: str | None = None) -> dict | None:
    mask = rules_in_network["service_category"] == service_category
    if scheme: mask &= rules_in_network["scheme_name"] == scheme
    pool = rules_in_network[mask]
    return None if pool.empty else pool.sample(1).iloc[0].to_dict()

def make_training_record(messages, metadata) -> dict:
    return {"tools": RANGA_TOOLS, "messages": messages, "metadata": metadata}

## 8 — Generate SFT pairs (function-calling)

### 8a — Template trajectories (300) — deterministic

In [ ]:
QUERY_TEMPLATES = [
    "I use {scheme}. I need {service}. Which covered clinic near ALU should I go to?",
    "I use {scheme} and have a referral from my health post. I need {service}. Where should I go?",
    "I use {scheme}. I have {service} needs and a long history with this condition. Do I still need a referral?",
    "My {scheme} claim at {prior} was rejected. Find a covered {service} provider near ALU.",
]

oon_rules = rules_full[rules_full["out_of_network"]==1]
prior_failure_map = {r["name"]: r["provider_id"] for _, r in oon_rules.drop_duplicates("name").iterrows()}

template_pairs = []
attempts = 0
while len(template_pairs) < N_TEMPLATE and attempts < N_TEMPLATE * 5:
    attempts += 1
    scheme, service = random.choice(SCHEMES), random.choice(SERVICES)
    rule = pick_rule(service, scheme)
    if rule is None: continue
    has_referral, has_chronic = random.random() > 0.3, random.random() > 0.6
    prior_id, prior_name = None, None
    if random.random() < 0.2 and prior_failure_map:
        prior_name = random.choice(list(prior_failure_map))
        prior_id = prior_failure_map[prior_name]
        query = QUERY_TEMPLATES[3].format(scheme=scheme, service=service.lower(), prior=prior_name)
    elif has_chronic:
        query = QUERY_TEMPLATES[2].format(scheme=scheme, service=service)
    elif has_referral:
        query = QUERY_TEMPLATES[1].format(scheme=scheme, service=service.lower())
    else:
        query = QUERY_TEMPLATES[0].format(scheme=scheme, service=service.lower())
    messages, tool_log = build_tool_trajectory(query, rule, has_referral, has_chronic, prior_id)
    template_pairs.append(make_training_record(messages, {
        "type": "template", "provider_id": rule["provider_id"], "scheme": scheme, "service": service,
        "bypass": has_chronic and int(rule["requires_referral"])==1,
        "oop_rwf": tool_log[-1]["result"]["oop_rwf"], "tool_calls": len(tool_log),
        "prior_failure": prior_name,
    }))
print(f"Generated {len(template_pairs)} template trajectories.")

### 8b — Custom trajectories (200) — LLM writes the user query only

In [ ]:
from tqdm.notebook import tqdm

SAFETY_CHECK = re.compile(
    r'\b(diagnos\w+|prescri\w+|antibiotic|ibuprofen|paracetamol|amoxicillin|'
    r'metformin|medication|drug|tablet|mg |ml |injection|biopsy|surgery|'
    r'you have|you may have|this could be|sounds like a)\b', re.IGNORECASE)

def generate_user_query(ctx: dict, rule: dict, prior_failure: str | None) -> str | None:
    hint = f"Their last claim at {prior_failure} was rejected. " if prior_failure else ""
    prompt = textwrap.dedent(f"""\
        Write ONE short spoken sentence a Rwandan university student would say to a hospital routing app.
        - Insurance: {rule['scheme_name']}
        - Service needed: {rule['service_category']}
        - Age band: {ctx['age_band']}
        - Context: {ctx['complaint_seed']}
        {hint}
        Do NOT mention specific hospital names or prices. No diagnosis. No drug names.
        Reply with only the query sentence.
        """).strip()
    raw = llm(prompt, max_tokens=80, temp=TEMPERATURE).strip().strip('"')
    return None if SAFETY_CHECK.search(raw) or len(raw) < 15 else raw

oon_names = list(prior_failure_map.keys()) or ["Legacy Hospital"]
custom_pairs, ctx_pool = [], contexts.copy()
random.shuffle(ctx_pool)
pbar = tqdm(total=N_CUSTOM, desc="Custom trajectories")
idx = 0
while len(custom_pairs) < N_CUSTOM and idx < len(ctx_pool) * 4:
    ctx = ctx_pool[idx % len(ctx_pool)]; idx += 1
    rule = pick_rule(ctx["service_category"])
    if rule is None: continue
    prior_name = random.choice(oon_names) if random.random() < 0.25 else None
    prior_id = prior_failure_map.get(prior_name) if prior_name else None
    query = generate_user_query(ctx, rule, prior_name)
    if query is None: continue
    has_referral = "referral" in query.lower()
    has_chronic = ctx["has_chronic"] or "history" in query.lower() or "long time" in query.lower()
    messages, tool_log = build_tool_trajectory(query, rule, has_referral, has_chronic, prior_id)
    if SAFETY_CHECK.search(messages[-1]["content"]): continue
    custom_pairs.append(make_training_record(messages, {
        "type": "custom", "source_row_id": ctx["source_row_id"],
        "provider_id": rule["provider_id"], "scheme": rule["scheme_name"],
        "service": rule["service_category"],
        "bypass": has_chronic and int(rule["requires_referral"])==1,
        "oop_rwf": tool_log[-1]["result"]["oop_rwf"], "tool_calls": len(tool_log),
        "prior_failure": prior_name,
    }))
    pbar.update(1)
pbar.close()
print(f"Generated {len(custom_pairs)} custom trajectories.")

### 8c — Save SFT file

In [ ]:
sft_pairs = template_pairs + custom_pairs
random.shuffle(sft_pairs)
with open(SFT_PATH, "w") as f:
    for rec in sft_pairs:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
sample = random.choice(sft_pairs)
print(f"Saved {len(sft_pairs)} records to {SFT_PATH}")
print(f"\nExample trajectory ({sample['metadata']['tool_calls']} tool calls):")
for m in sample["messages"]:
    role = m["role"]
    if role == "assistant" and m.get("tool_calls"):
        for tc in m["tool_calls"]:
            print(f"  [assistant] → {tc['function']['name']}({tc['function']['arguments']})")
    elif role == "tool":
        print(f"  [tool] ← {m['name']}: {m['content'][:80]}…")
    elif role == "assistant" and m.get("content"):
        print(f"  [assistant] {m['content'][:120]}…")

## 9 — Generate DPO preference pairs (tool-calling)

Rejected trajectories teach the model what **not** to do with tools.

In [ ]:
DPO_DISTRIBUTION = {
    "skipped_tool_calls":   60,  # answers without calling tools
    "out_of_network":       50,  # recommends OON provider despite tool result
    "wrong_tier":           40,  # ignores referral requirement from tool
    "missing_pre_auth":     30,  # omits pre-auth after tool flagged it
    "hallucinated_cost":    20,  # states wrong OOP ignoring calculate_oop result
}
assert sum(DPO_DISTRIBUTION.values()) == N_DPO

def corrupt_trajectory(chosen: dict, flaw: str) -> dict | None:
    """Build a rejected trajectory by corrupting the chosen tool-use pattern."""
    rejected = copy.deepcopy(chosen)
    msgs = rejected["messages"]
    final = msgs[-1]["content"]
    meta = chosen["metadata"]
    oop = meta["oop_rwf"]

    if flaw == "skipped_tool_calls":
        # Keep only system + user + a direct answer (no tools)
        rejected["messages"] = [msgs[0], msgs[1], {"role": "assistant", "content": final}]
        rejected["metadata"]["flaw"] = "skipped_tool_calls"
        return rejected

    if flaw == "out_of_network":
        oon_name = meta.get("prior_failure") or "Legacy Hospital"
        rejected["messages"][-1]["content"] = final.replace(
            re.search(r'^[^.]+', final).group(), f"Go to {oon_name}"
        )
        rejected["metadata"]["flaw"] = "out_of_network"
        return rejected

    if flaw == "wrong_tier":
        rejected["messages"][-1]["content"] = (
            "You can go directly to the hospital without a referral. " + final
        )
        rejected["metadata"]["flaw"] = "wrong_tier"
        return rejected

    if flaw == "missing_pre_auth":
        rejected["messages"][-1]["content"] = re.sub(
            r'Obtain pre-authorisation.*?\.\s*', '', final, flags=re.I)
        rejected["metadata"]["flaw"] = "missing_pre_auth"
        return rejected

    if flaw == "hallucinated_cost":
        rejected["messages"][-1]["content"] = re.sub(
            rf'{oop:,}', f'{oop//2:,}', final)
        rejected["metadata"]["flaw"] = "hallucinated_cost"
        return rejected

    return None

dpo_pairs = []
sft_pool = [p for p in sft_pairs if p["metadata"].get("type") == "template"]
random.shuffle(sft_pool)
pool_idx = 0
for flaw, target in DPO_DISTRIBUTION.items():
    collected = 0
    pbar = tqdm(total=target, desc=flaw)
    while collected < target and pool_idx < len(sft_pool) * 3:
        chosen_rec = sft_pool[pool_idx % len(sft_pool)]; pool_idx += 1
        if flaw == "missing_pre_auth":
            if "pre-authorisation" not in chosen_rec["messages"][-1]["content"]: continue
        rejected_rec = corrupt_trajectory(chosen_rec, flaw)
        if rejected_rec is None: continue
        dpo_pairs.append({
            "tools": RANGA_TOOLS,
            "chosen": chosen_rec,
            "rejected": rejected_rec,
            "rejection_type": flaw,
            "metadata": chosen_rec["metadata"],
        })
        collected += 1; pbar.update(1)
    pbar.close()
print(f"Total DPO pairs: {len(dpo_pairs)}")

In [ ]:
with open(DPO_PATH, "w") as f:
    for pair in dpo_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")
from collections import Counter
print(f"Saved {len(dpo_pairs)} DPO pairs to {DPO_PATH}")
for k, v in Counter(p["rejection_type"] for p in dpo_pairs).most_common():
    print(f"  {k:<22} {v}")

## 10 — Generate evaluation queries (50)

Each eval record includes the **expected tool-call sequence** for automated scoring.

In [ ]:
VOICE_TEMPLATES = [
    "I use {scheme}. I need {service}. Where should I go near ALU?",
    "My insurance is {scheme} and I have {service} needs. Which clinic is covered?",
    "I need {service} — I am on {scheme}. What is my cheapest covered option?",
]

eval_queries, attempts = [], 0
while len(eval_queries) < N_EVAL and attempts < N_EVAL * 10:
    attempts += 1
    scheme, service = random.choice(SCHEMES), random.choice(SERVICES)
    rule = pick_rule(service, scheme)
    if rule is None: continue
    has_chronic = random.random() > 0.6
    bypass = has_chronic and int(rule["requires_referral"]) == 1
    query = random.choice(VOICE_TEMPLATES).format(scheme=scheme, service=service.lower())
    if has_chronic: query += " I have been dealing with this for a long time."
    expected_tools = [
        {"name": "search_covered_providers", "arguments": {"scheme": scheme, "service": service}},
        {"name": "calculate_oop_cost", "arguments": {
            "provider_id": rule["provider_id"], "scheme": scheme, "service": service,
            "referral_followed": not bypass, "chronic_bypass": bypass,
        }},
    ]
    oop = compute_oop(rule, referral_followed=(not bypass))
    eval_queries.append({
        "query": query,
        "tools": RANGA_TOOLS,
        "expected_tool_calls": expected_tools,
        "expected_service": service,
        "expected_provider": rule["provider_id"],
        "expected_scheme": scheme,
        "expected_bypass": bypass,
        "expected_pre_auth": int(rule["requires_pre_auth"]) == 1,
        "expected_oop_rwf": oop,
        "ground_truth_label": "correct_tool_trajectory",
    })

with open(EVAL_PATH, "w") as f:
    for q in eval_queries:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")
print(f"Saved {len(eval_queries)} eval records to {EVAL_PATH}")

## 11 — Validation report

In [ ]:
def count_tool_violations(path: pathlib.Path) -> dict:
    no_tools, no_calc = 0, 0
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            msgs = rec.get("messages", rec.get("chosen", {}).get("messages", []))
            tool_names = []
            for m in msgs:
                if m.get("tool_calls"):
                    tool_names += [tc["function"]["name"] for tc in m["tool_calls"]]
            if not tool_names: no_tools += 1
            if "calculate_oop_cost" not in tool_names: no_calc += 1
    return {"missing_any_tool": no_tools, "missing_calculate_oop": no_calc}

def count_clinical_violations(path: pathlib.Path) -> int:
    n = 0
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            for key in ("messages",):
                if key in rec:
                    text = " ".join(m.get("content") or "" for m in rec[key])
                    if SAFETY_CHECK.search(text): n += 1
            if "chosen" in rec:
                text = " ".join(m.get("content") or "" for m in rec["chosen"]["messages"])
                if SAFETY_CHECK.search(text): n += 1
    return n

def line_count(p): return sum(1 for _ in open(p)) if p.exists() else 0

print("=" * 55)
print("VALIDATION REPORT — Function-Calling Dataset")
print("=" * 55)
print(f"Model           : {MODEL_ID}")
print(f"Providers       : {len(hospitals)} from {GEOJSON_PATH.name}")
print(f"Tools           : {len(RANGA_TOOLS)} → {TOOLS_PATH.name}")
print(f"In-network rules: {len(rules_in_network)}")
print()
sft_v = count_tool_violations(SFT_PATH)
print(f"SFT   lines={line_count(SFT_PATH)}  missing_tools={sft_v['missing_any_tool']}  missing_oop_call={sft_v['missing_calculate_oop']}")
print(f"      clinical_violations={count_clinical_violations(SFT_PATH)}")
dpo_v = count_tool_violations(DPO_PATH)
print(f"DPO   lines={line_count(DPO_PATH)}  chosen_missing_tools={dpo_v['missing_any_tool']}")
print(f"Eval  lines={line_count(EVAL_PATH)}")
print()
avg_tools = sum(r["metadata"]["tool_calls"] for r in sft_pairs) / len(sft_pairs)
print(f"Avg tool calls per SFT record: {avg_tools:.1f}")
print(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M')}")